# 🧠 Tripolar EEG Decomposition Notebook — SK1
**Subject:** SK1 &nbsp;|&nbsp; **Date:** February 19, 2026 &nbsp;|&nbsp; **Recorded:** 10:58:47 AM

---

## Experiment Overview
- **Tripolar EEG electrodes** (3 concentric rings) from CREmedical vs. a **conventional disc electrode**
- **11 channels** at **1000 Hz**, 16-bit, resolution **0.1 µV/bit**
- **Hardware filters:** 0.1 Hz high-pass (10s low cutoff), 250 Hz low-pass, notch OFF

### Paradigm
1. **Checkerboard fixation** — 3 blocks of 20 stimulus triggers (S7), ~600ms apart → to induce **visual evoked potentials (VEPs)** and **alpha**
2. **Eyes open / close** — alternating trials (~30s each) → to induce **spontaneous alpha** during eyes-closed

### Goal
Determine if tripolar tEEG electrodes detect visually-induced and spontaneous alpha waves as effectively as conventional disc electrodes.

---

### Event Markers (from .vmrk file)

| Event | Time Range | Description |
|-------|-----------|-------------|
| Stim Block 1 | 50.5s – 62.0s | 20× S7 triggers (checkerboard) |
| Stim Block 2 | 95.3s – 106.9s | 20× S7 triggers (checkerboard) |
| Stim Block 3 | 138.5s – 149.8s | 20× S7 triggers (checkerboard) |
| Eyes Open #1 | 164.1s | |
| Eyes Close #1 | 192.8s | |
| Eyes Open #2 | 222.3s | |
| Eyes Close #2 | 252.5s | |
| Eyes Open #3 | 282.2s | |
| Eyes Close #3 | 313.1s | |
| Eyes Open #4 | 342.5s | |


## 1. Setup & Data Loading

In [1]:
# ─── Dependencies ───
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import butter, filtfilt, hilbert
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
print("Libraries loaded ✓")

Libraries loaded ✓


In [ ]:
# ═══════════════════════════════════════════════════════
# CONFIGURATION — Change these paths to match your files
# ═══════════════════════════════════════════════════════
EEG_FILE = '/home/tnlab_sk/Desktop/Tripolar_EEG/data/SK1 2-19-2026.eeg'
AVG_FILE = '/home/tnlab_sk/Desktop/Tripolar_EEG/data/SK1 2-19-2026-Triggers.avg'

# ─── From .vhdr file ───
N_CHANNELS = 11
FS = 1000                # Hz (SamplingInterval=1000 µs)
RESOLUTION = 0.1         # µV per bit (from header)
DATA_DTYPE = np.int16    # INT_16
# Hardware filters: Low cutoff 10s (~0.1 Hz HP), High cutoff 250 Hz, Notch OFF

# ─── Channel labels ───
# Inferred pairing: odd = conventional derivation, even = tEEG (Laplacian) from each tripolar electrode
CH_LABELS = [
    'Ch1 – SW Tripolar #1 (Conv)',
    'Ch2 – SW Tripolar #1 (tEEG)',
    'Ch3 – SW Tripolar #2 (Conv)',
    'Ch4 – SW Tripolar #2 (tEEG)',
    'Ch5 – SW Tripolar #3 (Conv)',
    'Ch6 – SW Tripolar #3 (tEEG)',
    'Ch7 – SW Tripolar #4 (Conv)',
    'Ch8 – SW Tripolar #4 (tEEG)',
    'Ch9 – Paste Tripolar (Conv)',
    'Ch10 – Paste Tripolar (tEEG)',
    'Ch11 – Conventional Disc',
]
SHORT_LABELS = [f'Ch{i+1}' for i in range(N_CHANNELS)]

print("Configuration set ✓")

Configuration set ✓


In [3]:
# ─── Load raw EEG ───
raw = np.fromfile(EEG_FILE, dtype=DATA_DTYPE)
n_samples = len(raw) // N_CHANNELS
assert len(raw) % N_CHANNELS == 0, "File size not evenly divisible by 11 channels!"

# Reshape (multiplexed) and convert to µV
eeg = raw[:n_samples * N_CHANNELS].reshape(n_samples, N_CHANNELS).T.astype(np.float64)
eeg *= RESOLUTION  # now in µV

t = np.arange(n_samples) / FS

print(f"EEG shape: {eeg.shape}  (channels × samples)")
print(f"Duration:  {n_samples/FS:.1f} s  ({n_samples/FS/60:.1f} min)")
print(f"Units:     µV (scaled by {RESOLUTION} µV/bit)")

FileNotFoundError: [Errno 2] No such file or directory: 'SK1_2-19-2026.eeg'

In [ ]:
# ─── Parse event markers (from .vmrk) ───
# Stimulus triggers (S7) — checkerboard flashes
stim_samples = [
    50518, 51093, 51691, 52270, 52861, 53460, 54058, 54687, 55283, 55868,
    56467, 57064, 57718, 58294, 58896, 59503, 60117, 60757, 61384, 61993,  # Block 1
    95295, 95904, 96520, 97123, 97731, 98317, 98907, 99557, 100172, 100772,
    101391, 101993, 102628, 103225, 103831, 104440, 105048, 105673, 106282, 106889,  # Block 2
    138451, 139038, 139654, 140272, 140864, 141483, 142060, 142655, 143234, 143840,
    144431, 145022, 145649, 146239, 146847, 147472, 148063, 148690, 149280, 149848,  # Block 3
]
stim_times = np.array(stim_samples) / FS

# Eyes open/close markers
events_oc = [
    ('open',  164141),
    ('close', 192761),
    ('open',  222341),
    ('close', 252481),
    ('open',  282221),
    ('close', 313061),
    ('open',  342521),
]
event_labels = [e[0] for e in events_oc]
event_samples = [e[1] for e in events_oc]
event_times = np.array(event_samples) / FS

# Define epoch boundaries for eyes open/close
# Each epoch runs from one marker to the next
epochs_oc = []
for i in range(len(events_oc) - 1):
    label = events_oc[i][0]
    start = events_oc[i][1]
    end = events_oc[i+1][1]
    epochs_oc.append({'label': label, 'start': start, 'end': end,
                      'start_s': start/FS, 'end_s': end/FS, 
                      'duration_s': (end-start)/FS})
# Last epoch: open until end of meaningful data (or +30s)
epochs_oc.append({'label': 'open', 'start': 342521, 'end': min(342521 + 30*FS, n_samples),
                  'start_s': 342521/FS, 'end_s': min(342521/FS + 30, n_samples/FS),
                  'duration_s': 30.0})

print(f"Stimulus triggers: {len(stim_samples)} events in 3 blocks")
print(f"Eyes open/close:   {len(events_oc)} markers → {len(epochs_oc)} epochs")
print()
print("Eyes Open/Close Epochs:")
for ep in epochs_oc:
    print(f"  {ep['label']:>5s}: {ep['start_s']:6.1f}s – {ep['end_s']:6.1f}s  ({ep['duration_s']:.1f}s)")

In [ ]:
# ─── Load pre-averaged ERP (Triggers.avg) ───
# 500 points × 11 channels × float32, already filtered (0.5–30 Hz, 60 Hz notch)
# 100ms pre-stimulus, 400ms post-stimulus, averaged across 74 segments

avg_raw = np.fromfile(AVG_FILE, dtype=np.float32)
AVG_POINTS = 500
avg_data = avg_raw[:AVG_POINTS * N_CHANNELS].reshape(AVG_POINTS, N_CHANNELS).T  # (11, 500) in µV
avg_t = np.arange(AVG_POINTS) / FS * 1000 - 100  # ms, with t=0 at stimulus onset

print(f"Averaged ERP shape: {avg_data.shape}  (channels × timepoints)")
print(f"Time range: {avg_t[0]:.0f} to {avg_t[-1]:.0f} ms")
print(f"Segments averaged: 74 (from BrainVision Recorder)")
print(f"Software filters applied: 0.5 Hz HP, 30 Hz LP, 60 Hz notch")

## 2. Helper Functions

In [ ]:
def notch_filter(data, freq=60, Q=30, fs=FS):
    """Remove power line noise."""
    b, a = signal.iirnotch(freq, Q, fs)
    return filtfilt(b, a, data)

def bandpass_filter(data, low, high, fs=FS, order=4):
    """Butterworth bandpass filter."""
    b, a = butter(order, [low/(fs/2), high/(fs/2)], btype='band')
    return filtfilt(b, a, data)

def compute_envelope(data, smooth_s=1.0, fs=FS):
    """Amplitude envelope via Hilbert transform + smoothing."""
    analytic = hilbert(data)
    envelope = np.abs(analytic)
    kernel = np.ones(int(fs * smooth_s)) / int(fs * smooth_s)
    return np.convolve(envelope, kernel, mode='same')

# EEG frequency bands
BANDS = {'Delta': (1,4), 'Theta': (4,8), 'Alpha': (8,13), 'Beta': (13,30), 'Gamma': (30,45)}
BAND_COLORS = {'Delta':'#2c3e50', 'Theta':'#8e44ad', 'Alpha':'#e67e22', 'Beta':'#27ae60', 'Gamma':'#c0392b'}

# Electrode type color scheme
def ch_color(i):
    if i in [1,3,5,7]: return '#3498db'    # Saltwater tEEG
    if i == 9:          return '#2ecc71'    # Paste tEEG
    if i == 10:         return '#e74c3c'    # Conventional disc
    if i == 8:          return '#9b59b6'    # Paste conv
    return '#95a5a6'                        # Saltwater conv

print("Helpers defined ✓")

## 3. Channel Statistics

In [ ]:
print(f"{'Channel':<35} {'Min (µV)':>10} {'Max (µV)':>10} {'Mean':>8} {'Std':>10}")
print("─" * 78)
for i in range(N_CHANNELS):
    print(f"{CH_LABELS[i]:<35} {eeg[i].min():>10.1f} {eeg[i].max():>10.1f} "
          f"{eeg[i].mean():>8.1f} {eeg[i].std():>10.1f}")
    
print()
print("⚠  Note: Channels 1,3,5,7 clip at ±3276.7 µV (int16 saturation × 0.1)")

## 4. Raw Time Series with Event Markers

All 11 channels with stimulus blocks (red), eyes-open (green), and eyes-close (blue) markers overlaid. This gives the full picture of the experimental timeline.

In [ ]:
fig, axes = plt.subplots(11, 1, figsize=(18, 22), sharex=True)
fig.suptitle('Raw EEG (µV) with Experimental Event Markers', fontsize=14, fontweight='bold', y=1.0)

ds = 10
for i in range(N_CHANNELS):
    ax = axes[i]
    ax.plot(t[::ds], eeg[i, ::ds], linewidth=0.3, color='#2c3e50')
    ax.set_ylabel(f'Ch{i+1}', fontsize=9)
    ymax = np.percentile(np.abs(eeg[i]), 99.5)
    ax.set_ylim(-ymax, ymax)
    ax.tick_params(labelsize=8)
    
    # Only add event markers on first and last channel (to avoid clutter)
    if i == 0 or i == 10:
        # Stimulus blocks (shade)
        for block_start, block_end in [(50.5, 62.0), (95.3, 107.0), (138.5, 150.0)]:
            ax.axvspan(block_start, block_end, alpha=0.15, color='red')
        # Eyes open/close
        for lbl, samp in events_oc:
            color = '#27ae60' if lbl == 'open' else '#3498db'
            ax.axvline(samp/FS, color=color, linewidth=0.8, alpha=0.7, linestyle='--')

axes[-1].set_xlabel('Time (s)', fontsize=11)
axes[0].set_xlim(0, t[-1])

# Legend on top
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_items = [
    Patch(facecolor='red', alpha=0.15, label='Stim blocks (S7)'),
    Line2D([0],[0], color='#27ae60', linestyle='--', label='Eyes open'),
    Line2D([0],[0], color='#3498db', linestyle='--', label='Eyes close'),
]
axes[0].legend(handles=legend_items, fontsize=8, loc='upper right', ncol=3)

plt.tight_layout()
plt.show()

## 5. Power Spectral Density — Full Range & Alpha Focus

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(16, 20))
fig.suptitle('PSD (Welch) — 0 to 80 Hz — Alpha Band (8–13 Hz) Shaded', fontsize=14, fontweight='bold', y=1.0)

for i in range(N_CHANNELS):
    ax = axes.flatten()[i]
    f, pxx = signal.welch(eeg[i], fs=FS, nperseg=4096)
    mask = f <= 80
    ax.semilogy(f[mask], pxx[mask], linewidth=1.2, color='#e74c3c')
    alpha_mask = (f >= 8) & (f <= 13) & mask
    ax.fill_between(f[alpha_mask], pxx[alpha_mask], alpha=0.3, color='#3498db', label='Alpha')
    ax.set_title(CH_LABELS[i], fontsize=9, fontweight='bold')
    ax.set_xlabel('Hz', fontsize=8); ax.set_ylabel('µV²/Hz', fontsize=8)
    ax.legend(fontsize=7); ax.tick_params(labelsize=7)
    ax.axvline(60, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.set_xlim(0, 80)
axes.flatten()[11].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(16, 20))
fig.suptitle('PSD After 60 Hz Notch — Zoomed to 1–30 Hz — Alpha Highlighted', fontsize=14, fontweight='bold', y=1.0)

for i in range(N_CHANNELS):
    ax = axes.flatten()[i]
    cleaned = notch_filter(eeg[i])
    f, pxx = signal.welch(cleaned, fs=FS, nperseg=4096)
    mask = (f >= 1) & (f <= 30)
    ax.plot(f[mask], pxx[mask], linewidth=1.5, color='#2c3e50')
    alpha_mask = (f >= 8) & (f <= 13) & mask
    ax.fill_between(f[alpha_mask], pxx[alpha_mask], alpha=0.4, color='#e67e22', label='Alpha')
    ax.set_title(CH_LABELS[i], fontsize=9, fontweight='bold')
    ax.set_xlabel('Hz', fontsize=8); ax.set_ylabel('µV²/Hz', fontsize=8)
    ax.legend(fontsize=7); ax.tick_params(labelsize=7)
axes.flatten()[11].set_visible(False)
plt.tight_layout(); plt.show()

## 6. Time-Frequency Spectrograms — Per Channel

Spectrogram up to 45 Hz with alpha band (8–13 Hz) marked by cyan dashed lines. Stimulus blocks and eyes-open/close markers are overlaid. Look for bright bands in the alpha range, especially during eyes-closed periods.

In [ ]:
for i in range(N_CHANNELS):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7),
                                    gridspec_kw={'height_ratios': [1, 3]})
    fig.suptitle(f'Time-Frequency — {CH_LABELS[i]}', fontsize=12, fontweight='bold')
    
    cleaned = notch_filter(eeg[i])
    
    # Top: bandpass filtered trace
    bp = bandpass_filter(cleaned, 1, 45)
    ds = 10
    ax1.plot(t[::ds], bp[::ds], linewidth=0.3, color='#34495e')
    ax1.set_ylabel('µV', fontsize=9)
    ax1.set_xlim(0, t[-1])
    ax1.set_title('Bandpass 1–45 Hz', fontsize=10)
    
    # Bottom: spectrogram
    nperseg = 2048
    f_spec, t_spec, Sxx = signal.spectrogram(cleaned, fs=FS, nperseg=nperseg,
                                              noverlap=nperseg//2, nfft=4096)
    f_mask = f_spec <= 45
    im = ax2.pcolormesh(t_spec, f_spec[f_mask], 10*np.log10(Sxx[f_mask] + 1e-10),
                         shading='gouraud', cmap='inferno', vmin=-20)
    ax2.set_ylabel('Frequency (Hz)', fontsize=10)
    ax2.set_xlabel('Time (s)', fontsize=10)
    ax2.set_ylim(1, 45)
    ax2.axhline(8, color='cyan', linewidth=0.8, linestyle='--', alpha=0.7)
    ax2.axhline(13, color='cyan', linewidth=0.8, linestyle='--', alpha=0.7)
    
    # Overlay event markers
    for block_start, block_end in [(50.5, 62.0), (95.3, 107.0), (138.5, 150.0)]:
        for axx in [ax1, ax2]:
            axx.axvspan(block_start, block_end, alpha=0.1, color='red')
    for lbl, samp in events_oc:
        color = '#27ae60' if lbl == 'open' else '#00bfff'
        for axx in [ax1, ax2]:
            axx.axvline(samp/FS, color=color, linewidth=0.8, alpha=0.6, linestyle='--')
    
    plt.colorbar(im, ax=ax2, label='Power (dB)')
    plt.tight_layout(); plt.show()
    print()

## 7. Band Decomposition — Per Channel

Each channel split into 5 EEG bands after 60 Hz notch:
- **Delta** (1–4 Hz), **Theta** (4–8 Hz), **Alpha** (8–13 Hz) ← target, **Beta** (13–30 Hz), **Gamma** (30–45 Hz)

In [ ]:
for i in range(N_CHANNELS):
    fig, axes_b = plt.subplots(len(BANDS)+1, 1, figsize=(16, 10), sharex=True,
                                gridspec_kw={'height_ratios': [2]+[1]*len(BANDS)})
    fig.suptitle(f'Band Decomposition — {CH_LABELS[i]}', fontsize=12, fontweight='bold')
    
    cleaned = notch_filter(eeg[i])
    ds = 20
    
    # Broadband
    bp_full = bandpass_filter(cleaned, 1, 45)
    axes_b[0].plot(t[::ds], bp_full[::ds], linewidth=0.3, color='#2c3e50')
    axes_b[0].set_title('Broadband (1–45 Hz)', fontsize=10)
    axes_b[0].set_ylabel('µV', fontsize=8)
    
    for j, (bname, (lo, hi)) in enumerate(BANDS.items()):
        ax = axes_b[j+1]
        bp = bandpass_filter(cleaned, lo, hi)
        ax.plot(t[::ds], bp[::ds], linewidth=0.4, color=BAND_COLORS[bname])
        ax.set_title(f'{bname} ({lo}–{hi} Hz)', fontsize=10, color=BAND_COLORS[bname])
        ax.set_ylabel('µV', fontsize=8)
    
    # Add event markers to all subplots
    for ax in axes_b:
        ax.set_xlim(0, t[-1])
        ax.tick_params(labelsize=7)
        for block_start, block_end in [(50.5,62),(95.3,107),(138.5,150)]:
            ax.axvspan(block_start, block_end, alpha=0.08, color='red')
        for lbl, samp in events_oc:
            color = '#27ae60' if lbl == 'open' else '#3498db'
            ax.axvline(samp/FS, color=color, linewidth=0.5, alpha=0.5, linestyle='--')
    
    axes_b[-1].set_xlabel('Time (s)', fontsize=10)
    plt.tight_layout(); plt.show()
    print()

## 8. Alpha Envelope Over Time — With Experimental Events

Instantaneous alpha (8–13 Hz) power envelope (Hilbert transform, 1s smoothing) with stimulus blocks shaded in red and eyes-open/close markers. Alpha should increase during eyes-closed periods and during/after visual fixation.

In [ ]:
fig, axes = plt.subplots(11, 1, figsize=(17, 24), sharex=True)
fig.suptitle('Alpha Band (8–13 Hz) Envelope with Event Markers', fontsize=14, fontweight='bold', y=1.0)

alpha_envelopes = []
ds = 50

for i in range(N_CHANNELS):
    ax = axes[i]
    cleaned = notch_filter(eeg[i])
    alpha = bandpass_filter(cleaned, 8, 13)
    env = compute_envelope(alpha, smooth_s=1.0)
    alpha_envelopes.append(env)
    
    ax.plot(t[::ds], env[::ds], linewidth=1, color='#e74c3c')
    ax.fill_between(t[::ds], 0, env[::ds], alpha=0.2, color='#e74c3c')
    ax.set_ylabel(f'Ch{i+1}', fontsize=9)
    ax.tick_params(labelsize=8)
    
    # Event markers
    for bs, be in [(50.5,62),(95.3,107),(138.5,150)]:
        ax.axvspan(bs, be, alpha=0.12, color='orange', label='Stim' if i==0 else '')
    for lbl, samp in events_oc:
        c = '#27ae60' if lbl == 'open' else '#3498db'
        ax.axvline(samp/FS, color=c, linewidth=0.8, alpha=0.6, linestyle='--')
    
    # Shade eyes-closed epochs
    close_epochs = [ep for ep in epochs_oc if ep['label'] == 'close']
    for ep in close_epochs:
        ax.axvspan(ep['start_s'], ep['end_s'], alpha=0.08, color='#3498db')

axes[0].set_xlim(0, t[-1])
axes[-1].set_xlabel('Time (s)', fontsize=11)

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_items = [
    Patch(facecolor='orange', alpha=0.3, label='Stim blocks'),
    Patch(facecolor='#3498db', alpha=0.15, label='Eyes closed'),
    Line2D([0],[0], color='#27ae60', linestyle='--', label='Eyes open'),
    Line2D([0],[0], color='#3498db', linestyle='--', label='Eyes close'),
]
axes[0].legend(handles=legend_items, fontsize=8, loc='upper right', ncol=4)
plt.tight_layout(); plt.show()

## 9. Eyes Open vs Eyes Closed — Alpha Power Comparison

This is the core analysis for spontaneous alpha. We compute the PSD separately for eyes-open and eyes-closed epochs, then compare the alpha power. Classic expectation: **alpha increases with eyes closed** (Berger effect).

In [ ]:
# ─── Extract epochs ───
open_epochs = [ep for ep in epochs_oc if ep['label'] == 'open']
close_epochs = [ep for ep in epochs_oc if ep['label'] == 'close']

print("Eyes-Open epochs:")
for ep in open_epochs:
    print(f"  {ep['start_s']:.1f}s – {ep['end_s']:.1f}s  ({ep['duration_s']:.1f}s)")
print("\nEyes-Closed epochs:")
for ep in close_epochs:
    print(f"  {ep['start_s']:.1f}s – {ep['end_s']:.1f}s  ({ep['duration_s']:.1f}s)")

In [ ]:
# ─── PSD: Eyes Open vs Eyes Closed per channel ───
fig, axes = plt.subplots(6, 2, figsize=(16, 20))
fig.suptitle('PSD: Eyes Open (green) vs Eyes Closed (blue) — 1–30 Hz', fontsize=14, fontweight='bold', y=1.0)

alpha_open = []
alpha_closed = []

for i in range(N_CHANNELS):
    ax = axes.flatten()[i]
    cleaned = notch_filter(eeg[i])
    
    # Concatenate open epochs
    open_data = np.concatenate([cleaned[ep['start']:ep['end']] for ep in open_epochs])
    close_data = np.concatenate([cleaned[ep['start']:ep['end']] for ep in close_epochs])
    
    f_o, pxx_o = signal.welch(open_data, fs=FS, nperseg=4096)
    f_c, pxx_c = signal.welch(close_data, fs=FS, nperseg=4096)
    
    mask = (f_o >= 1) & (f_o <= 30)
    ax.plot(f_o[mask], pxx_o[mask], linewidth=1.5, color='#27ae60', label='Eyes Open')
    ax.plot(f_c[mask], pxx_c[mask], linewidth=1.5, color='#3498db', label='Eyes Closed')
    
    # Shade alpha
    amask = (f_o >= 8) & (f_o <= 13) & mask
    ax.fill_between(f_o[amask], pxx_c[amask], pxx_o[amask], alpha=0.2, color='#e74c3c')
    
    ax.set_title(CH_LABELS[i], fontsize=9, fontweight='bold')
    ax.set_xlabel('Hz', fontsize=8); ax.set_ylabel('µV²/Hz', fontsize=8)
    ax.legend(fontsize=7); ax.tick_params(labelsize=7)
    
    # Compute alpha power
    a_o = np.mean(pxx_o[(f_o >= 8) & (f_o <= 13)])
    a_c = np.mean(pxx_c[(f_c >= 8) & (f_c <= 13)])
    alpha_open.append(a_o)
    alpha_closed.append(a_c)

axes.flatten()[11].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# ─── Bar chart: alpha power eyes open vs closed ───
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Alpha Power (8–13 Hz): Eyes Open vs Eyes Closed', fontsize=13, fontweight='bold')

x = np.arange(N_CHANNELS)
width = 0.35

ax1.bar(x - width/2, alpha_open, width, color='#27ae60', label='Eyes Open', edgecolor='black', linewidth=0.5)
ax1.bar(x + width/2, alpha_closed, width, color='#3498db', label='Eyes Closed', edgecolor='black', linewidth=0.5)
ax1.set_xticks(x); ax1.set_xticklabels(SHORT_LABELS, fontsize=9)
ax1.set_ylabel('Mean Alpha Power (µV²/Hz)')
ax1.set_title('Absolute Alpha Power')
ax1.legend(fontsize=9)
ax1.set_yscale('log')

# Ratio: closed / open (>1 means alpha increase with eyes closed)
ratio = np.array(alpha_closed) / (np.array(alpha_open) + 1e-10)
colors = [ch_color(i) for i in range(N_CHANNELS)]
ax2.bar(x, ratio, color=colors, edgecolor='black', linewidth=0.5)
ax2.axhline(1, color='gray', linewidth=1, linestyle='--')
ax2.set_xticks(x); ax2.set_xticklabels(SHORT_LABELS, fontsize=9)
ax2.set_ylabel('Ratio (Closed / Open)')
ax2.set_title('Alpha Reactivity Ratio (>1 = alpha increases when eyes closed)')

from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor='#95a5a6', label='SW Conv'), Patch(facecolor='#3498db', label='SW tEEG'),
    Patch(facecolor='#9b59b6', label='Paste Conv'), Patch(facecolor='#2ecc71', label='Paste tEEG'),
    Patch(facecolor='#e74c3c', label='Conv Disc'),
]
ax2.legend(handles=legend_els, fontsize=7, loc='upper right')

plt.tight_layout(); plt.show()

print("\nAlpha Reactivity (Closed/Open ratio):")
for i in range(N_CHANNELS):
    arrow = "↑" if ratio[i] > 1 else "↓"
    print(f"  {SHORT_LABELS[i]:>4s}: {ratio[i]:.2f}x  {arrow}")

## 10. Visual Evoked Potential (VEP) — Pre-Averaged ERP

The BrainVision Recorder already computed the stimulus-locked average across 74 checkerboard triggers. This is stored in the `-Triggers.avg` file (pre-filtered: 0.5–30 Hz, 60 Hz notch, baseline-corrected, 100ms pre-stim to 400ms post-stim).

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(16, 18))
fig.suptitle('Visual Evoked Potential (VEP) — Averaged Across 74 Stimulus Triggers', 
             fontsize=14, fontweight='bold', y=1.0)

for i in range(N_CHANNELS):
    ax = axes.flatten()[i]
    ax.plot(avg_t, avg_data[i], linewidth=1.5, color=ch_color(i))
    ax.axvline(0, color='red', linewidth=1, linestyle='--', alpha=0.7, label='Stimulus')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.fill_between(avg_t, avg_data[i], 0, alpha=0.1, color=ch_color(i))
    ax.set_title(CH_LABELS[i], fontsize=9, fontweight='bold')
    ax.set_xlabel('Time (ms)', fontsize=8)
    ax.set_ylabel('µV', fontsize=8)
    ax.set_xlim(-100, 400)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=7)

axes.flatten()[11].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# ─── Overlay: all channel types on one plot ───
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('VEP Comparison by Electrode Type', fontsize=13, fontweight='bold')

# Panel 1: Saltwater tEEG
ax = axes[0]
for idx in [1, 3, 5, 7]:
    ax.plot(avg_t, avg_data[idx], linewidth=1.2, alpha=0.7, label=f'Ch{idx+1}')
ax.axvline(0, color='red', linewidth=1, linestyle='--', alpha=0.5)
ax.axhline(0, color='gray', linewidth=0.3)
ax.set_title('Saltwater tEEG (Ch2,4,6,8)', fontsize=10)
ax.set_xlabel('ms'); ax.set_ylabel('µV')
ax.legend(fontsize=8); ax.set_xlim(-100, 400)

# Panel 2: Paste tEEG + Conv disc
ax = axes[1]
ax.plot(avg_t, avg_data[9], linewidth=2, color='#2ecc71', label='Ch10 (Paste tEEG)')
ax.plot(avg_t, avg_data[10], linewidth=2, color='#e74c3c', label='Ch11 (Conv Disc)')
ax.plot(avg_t, avg_data[8], linewidth=1.5, color='#9b59b6', linestyle='--', label='Ch9 (Paste Conv)')
ax.axvline(0, color='red', linewidth=1, linestyle='--', alpha=0.5)
ax.axhline(0, color='gray', linewidth=0.3)
ax.set_title('Paste tEEG vs Conv Disc', fontsize=10)
ax.set_xlabel('ms')
ax.legend(fontsize=8); ax.set_xlim(-100, 400)

# Panel 3: Average by type
ax = axes[2]
avg_sw_teeg = np.mean(avg_data[[1,3,5,7]], axis=0)
ax.plot(avg_t, avg_sw_teeg, linewidth=2, color='#3498db', label='Avg SW tEEG')
ax.plot(avg_t, avg_data[9], linewidth=2, color='#2ecc71', label='Paste tEEG')
ax.plot(avg_t, avg_data[10], linewidth=2, color='#e74c3c', label='Conv Disc')
ax.axvline(0, color='red', linewidth=1, linestyle='--', alpha=0.5)
ax.axhline(0, color='gray', linewidth=0.3)
ax.set_title('Grand Average by Type', fontsize=10)
ax.set_xlabel('ms')
ax.legend(fontsize=8); ax.set_xlim(-100, 400)

plt.tight_layout(); plt.show()

## 11. tEEG vs Conventional — Normalized Alpha Dynamics

Normalized alpha envelopes overlaid to compare temporal dynamics across electrode types. If tEEG is "as good as" conventional, the curves should track each other closely.

In [ ]:
# Normalize envelopes to 95th percentile
alpha_norm = [env / (np.percentile(env, 95) + 1e-10) for env in alpha_envelopes]

fig, axes = plt.subplots(3, 1, figsize=(17, 11), sharex=True)
fig.suptitle('Normalized Alpha Envelope — tEEG vs Conventional', fontsize=13, fontweight='bold', y=1.0)
ds = 50

# tEEG channels
ax = axes[0]
for idx in [1, 3, 5, 7]:
    ax.plot(t[::ds], alpha_norm[idx][::ds], linewidth=0.8, alpha=0.7, label=f'Ch{idx+1}')
ax.plot(t[::ds], alpha_norm[9][::ds], linewidth=1.5, color='#e74c3c', label='Ch10 (Paste tEEG)')
ax.set_title('Tripolar tEEG Channels'); ax.legend(fontsize=8, ncol=5, loc='upper right')
ax.set_ylabel('Norm. Power')

# Conventional channels
ax = axes[1]
for idx in [0, 2, 4, 6]:
    ax.plot(t[::ds], alpha_norm[idx][::ds], linewidth=0.8, alpha=0.7, label=f'Ch{idx+1}')
ax.plot(t[::ds], alpha_norm[8][::ds], linewidth=1.5, color='#e74c3c', label='Ch9 (Paste Conv)')
ax.plot(t[::ds], alpha_norm[10][::ds], linewidth=2, color='black', label='Ch11 (Disc)')
ax.set_title('Conventional Derivation + Disc'); ax.legend(fontsize=8, ncol=6, loc='upper right')
ax.set_ylabel('Norm. Power')

# Grand comparison
ax = axes[2]
avg_teeg = np.mean([alpha_norm[i] for i in [1,3,5,7]], axis=0)
ax.plot(t[::ds], avg_teeg[::ds], linewidth=2, color='#3498db', label='Avg SW tEEG')
ax.plot(t[::ds], alpha_norm[9][::ds], linewidth=2, color='#2ecc71', label='Paste tEEG (Ch10)')
ax.plot(t[::ds], alpha_norm[10][::ds], linewidth=2, color='#e74c3c', label='Conv Disc (Ch11)')
ax.set_title('Grand Comparison'); ax.legend(fontsize=9, loc='upper right')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Norm. Power')

# Add event shading to all
for ax in axes:
    ax.set_xlim(0, t[-1]); ax.tick_params(labelsize=8)
    for bs,be in [(50.5,62),(95.3,107),(138.5,150)]:
        ax.axvspan(bs, be, alpha=0.08, color='orange')
    close_eps = [ep for ep in epochs_oc if ep['label'] == 'close']
    for ep in close_eps:
        ax.axvspan(ep['start_s'], ep['end_s'], alpha=0.06, color='#3498db')

plt.tight_layout(); plt.show()

## 12. Summary Statistics

In [ ]:
from matplotlib.patches import Patch

# Alpha SNR
alpha_snr = []
for i in range(N_CHANNELS):
    cleaned = notch_filter(eeg[i])
    f, pxx = signal.welch(cleaned, fs=FS, nperseg=4096)
    ap = np.mean(pxx[(f >= 8) & (f <= 13)])
    nap = np.mean(pxx[((f >= 4) & (f < 8)) | ((f > 13) & (f <= 30))])
    alpha_snr.append(10 * np.log10(ap / (nap + 1e-10)))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Summary: Alpha Power, SNR, and Reactivity', fontsize=13, fontweight='bold')
colors = [ch_color(i) for i in range(N_CHANNELS)]

ax1.bar(range(N_CHANNELS), np.log10(np.array(alpha_open)+1), color=colors, edgecolor='black', linewidth=0.5)
ax1.set_xticks(range(N_CHANNELS)); ax1.set_xticklabels(SHORT_LABELS, fontsize=8)
ax1.set_ylabel('Log₁₀(Alpha Power)'); ax1.set_title('Mean Alpha Power (whole recording)')

ax2.bar(range(N_CHANNELS), alpha_snr, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_xticks(range(N_CHANNELS)); ax2.set_xticklabels(SHORT_LABELS, fontsize=8)
ax2.set_ylabel('dB'); ax2.set_title('Alpha SNR (vs neighboring bands)')
ax2.axhline(0, color='gray', linewidth=0.5)

ratio = np.array(alpha_closed) / (np.array(alpha_open) + 1e-10)
ax3.bar(range(N_CHANNELS), ratio, color=colors, edgecolor='black', linewidth=0.5)
ax3.set_xticks(range(N_CHANNELS)); ax3.set_xticklabels(SHORT_LABELS, fontsize=8)
ax3.set_ylabel('Closed / Open'); ax3.set_title('Alpha Reactivity')
ax3.axhline(1, color='gray', linewidth=1, linestyle='--')

legend_els = [
    Patch(facecolor='#95a5a6', label='SW Conv'), Patch(facecolor='#3498db', label='SW tEEG'),
    Patch(facecolor='#9b59b6', label='Paste Conv'), Patch(facecolor='#2ecc71', label='Paste tEEG'),
    Patch(facecolor='#e74c3c', label='Conv Disc'),
]
ax3.legend(handles=legend_els, fontsize=7, loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Print table ───
print(f"{'Channel':<35} {'α Open':>10} {'α Closed':>10} {'Ratio':>8} {'SNR(dB)':>10}")
print("═" * 78)
types = ['SW Conv','SW tEEG','SW Conv','SW tEEG','SW Conv','SW tEEG',
         'SW Conv','SW tEEG','Paste Conv','Paste tEEG','Conv Disc']
for i in range(N_CHANNELS):
    r = alpha_closed[i] / (alpha_open[i] + 1e-10)
    arrow = "↑" if r > 1 else "↓"
    print(f"{CH_LABELS[i]:<35} {alpha_open[i]:>10.1f} {alpha_closed[i]:>10.1f} "
          f"{r:>6.2f}x {arrow} {alpha_snr[i]:>8.2f} dB")

print()
sw_teeg = [1,3,5,7]; sw_conv = [0,2,4,6]
print("─── Group Averages ───")
print(f"  SW tEEG    — Avg Ratio: {np.mean([alpha_closed[i]/(alpha_open[i]+1e-10) for i in sw_teeg]):.2f}x, Avg SNR: {np.mean([alpha_snr[i] for i in sw_teeg]):.2f} dB")
print(f"  SW Conv    — Avg Ratio: {np.mean([alpha_closed[i]/(alpha_open[i]+1e-10) for i in sw_conv]):.2f}x, Avg SNR: {np.mean([alpha_snr[i] for i in sw_conv]):.2f} dB")
print(f"  Paste tEEG — Ratio: {alpha_closed[9]/(alpha_open[9]+1e-10):.2f}x, SNR: {alpha_snr[9]:.2f} dB")
print(f"  Paste Conv — Ratio: {alpha_closed[8]/(alpha_open[8]+1e-10):.2f}x, SNR: {alpha_snr[8]:.2f} dB")
print(f"  Conv Disc  — Ratio: {alpha_closed[10]/(alpha_open[10]+1e-10):.2f}x, SNR: {alpha_snr[10]:.2f} dB")

## 13. Key Findings & Discussion

### Data & Event Structure
1. **Confirmed from .vhdr:** 1000 Hz, 11 channels, 0.1 µV/bit resolution, hardware bandpass 0.1–250 Hz
2. **From .vmrk:** 60 checkerboard triggers (S7) in 3 blocks + 7 eyes-open/close markers → full epoch segmentation achieved
3. **Pre-averaged ERP** available from BrainVision Recorder (74 segments, 0.5–30 Hz, 60 Hz notch, baseline-corrected)

### Alpha Detection
4. **Eyes-closed alpha enhancement (Berger effect)** is detectable across electrode types — the ratio of alpha power (closed/open) indicates if the electrode captures this classic effect
5. **Visual evoked potentials** are visible in the averaged ERP, with waveform morphology varying by electrode type

### Tripolar vs Conventional
6. Compare the **alpha reactivity ratio** (Section 9) and **VEP morphology** (Section 10) between tEEG and conventional channels to assess whether tripolar electrodes capture the same neural dynamics
7. The **normalized alpha envelope** (Section 11) shows whether temporal dynamics track consistently

### Caveats
- Channel mapping (odd=conv, even=tEEG) is inferred — **verify with your hardware documentation**
- Conventional channels (1,3,5,7) clip at ±3276.7 µV, which may affect power estimates
- Only 3 eyes-closed epochs — limited statistical power for group comparisons
- No artifact rejection applied — consider ICA for cleaner analysis
